# Assignment 2: Feedforward Neural Network (MNIST)
**Aim:** Implement a feedforward neural network with Keras and TensorFlow.

a. Import packages  b. Load data  c. Define network  d. Train with SGD  e. Evaluate  f. Plot loss & accuracy

## a. Import the necessary packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

I0000 00:00:1793640517.807962   35868 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793640519.568181   35868 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## b. Load the training and testing data (MNIST)

In [2]:
# Path of dataset folder (change if your folder is somewhere else)
train_dir = "dataset/mnist-jpg/train"
test_dir  = "dataset/mnist-jpg/test"

datagen = ImageDataGenerator(rescale=1.0/255)   # pixel values 0-255 -> 0-1

train_gen = datagen.flow_from_directory(train_dir, target_size=(28, 28),
                                        color_mode="grayscale", batch_size=60000,
                                        class_mode="categorical", shuffle=True)
test_gen  = datagen.flow_from_directory(test_dir, target_size=(28, 28),
                                        color_mode="grayscale", batch_size=10000,
                                        class_mode="categorical", shuffle=True)

# One big batch = whole dataset loaded as numpy arrays
x_train, y_train = train_gen[0]
x_test,  y_test  = test_gen[0]
class_names = list(train_gen.class_indices.keys())

print("x_train:", x_train.shape, " y_train:", y_train.shape)
print("x_test :", x_test.shape,  " y_test :", y_test.shape)
print("Classes:", class_names)

Found 60000 images belonging to 10 classes.


Found 10000 images belonging to 10 classes.


x_train: (60000, 28, 28, 1)  y_train: (60000, 10)
x_test : (10000, 28, 28, 1)  y_test : (10000, 10)
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [3]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[np.argmax(y_train[i])])
    plt.axis("off")
plt.show()

/tmp/ipykernel_35868/273738373.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## c. Define the network architecture
Flatten (28x28x1 = 784 inputs) → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax)

In [4]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

## d. Train the model using SGD

In [5]:
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(x_train, y_train,
                    validation_data=(x_test, y_test),
                    epochs=10, batch_size=64)

Epoch 1/10


  1/938 ━━━━━━━━━━━━━━━━━━━━ 4:25 284ms/step - accuracy: 0.1562 - loss: 2.2937

 29/938 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4057 - loss: 1.9998    

 60/938 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5943 - loss: 1.4756

 90/938 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6710 - loss: 1.1707

120/938 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7177 - loss: 1.0029

... (471 outputs left out)

888/938 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9932 - loss: 0.0278

907/938 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9932 - loss: 0.0278

926/938 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9931 - loss: 0.0279

938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9931 - loss: 0.0279 - val_accuracy: 0.9661 - val_loss: 0.1101


## e. Evaluate the network

In [6]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc*100:.2f}%")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d",
            xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

Test loss: 0.1101
Test accuracy: 96.61%


              precision    recall  f1-score   support

           0       0.98      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.99      0.90      0.95      1032
           3       0.89      0.99      0.94      1010
           4       0.98      0.96      0.97       982
           5       0.97      0.96      0.96       892
           6       0.97      0.97      0.97       958
           7       0.98      0.96      0.97      1028
           8       0.96      0.96      0.96       974
           9       0.95      0.98      0.96      1009

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000



/tmp/ipykernel_35868/4274663811.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
n = 25   # change index to test another image
plt.imshow(x_test[n].squeeze(), cmap="gray"); plt.axis("off"); plt.show()
pred = model.predict(x_test[n:n+1], verbose=0)
print("Predicted:", class_names[np.argmax(pred)])
print("Actual   :", class_names[np.argmax(y_test[n])])

Predicted: 6
Actual   : 6


/tmp/ipykernel_35868/3822332667.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(x_test[n].squeeze(), cmap="gray"); plt.axis("off"); plt.show()


## f. Plot the training loss and accuracy

In [8]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.show()

/tmp/ipykernel_35868/1834603888.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
